In [ ]:
!pip install -qU langchain langchain-openai langchain-community langchain-chroma chromadb pypdf
!pip install -q gradio

In [ ]:
import os
import re # regular expression: to find patterns in a string
from typing import List, Dict, Any, Optional, Pattern, Iterator
from IPython.display import display, Markdown
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
import json
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr
import shutil

In [ ]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
model = ChatOpenAI(model_name='gpt-4o-mini', temperature=0.0)

**Work Distribution**

Tan Zhiyu：


*   Stage 1: Ingestion and Indexing
*   Stage 2: Query Understanding



Chong Kai Jian


*   Stage 3: Retrieval
*   Stage 4: Context Refinement



Lim Zhi Kin


*   Stage 5: Generation
*   Stage 6: Interface



#Heading Structure
## Source Document Heading Structure
Document: 2024 National Money Laundering Risk Assessment
The source document is structured with the following top-level sections:

EXECUTIVE SUMMARY

INTRODUCTION

SECTION I. THREATS

SECTION II. VULNERABILITIES AND RISKS

CONCLUSION

PARTICIPANTS

METHODOLOGY

TERMINOLOGY

LIST OF ACRONYMS

Within these sections, the document contains mid-level topic headings (e.g., Fraud, Drug Trafficking, Cybercrime) and numbered subtopics (e.g., 1. Investment Fraud, 2. Cyber-Enabled Fraud). The ingestion pipeline detects all three levels and assigns metadata accordingly, enabling structure-aware retrieval through metadata filtering.

#Modifications & Design Rationale
## Modifications from Labs 6 and 7
Our team made several significant modifications to adapt the lab implementations to our source document and improve the overall RAG quality:
1. Restructured workflow architecture:
Lab 7 used a six-chain linear workflow (query compiler, analyzer chain 1, analyzer chain 2, parallel retriever, context compiler, responder). We restructured this into a cleaner four-stage pipeline (Query Understanding, Retrieval, Context Refinement, Generation), where each stage has a clearly defined input and output. This makes the workflow easier to debug and maintain.
2. Three-level heading detection (Stage 1):
Lab 6 used a two-level structure (section + subsection). Our ingestion pipeline detects three levels — top sections, topic headings, and numbered subtopics — to better match the hierarchical structure of our source document. The topic heading detection uses heuristics based on line length, capitalization ratio, and punctuation patterns.
3. LLM-based intent classification (Stage 2):
Lab 7 did not classify query intent. We added an LLM-based query understanding module that classifies user queries into predefined intent categories (e.g., threats, vulnerabilities, methodology) and maps them to relevant document sections. This enables more targeted metadata filtering during retrieval.
4. Query rewriting and expansion (Stage 2):
The query understanding module also rewrites user queries into retrieval-friendly forms and expands them with related terms to improve recall. Lab 7's query compiler only reformulated conversational queries into standalone queries.
5. Retrieval with automatic fallback (Stage 3):
Lab 7 used a parallel retriever with fixed metadata filters. Our retrieval stage first attempts a filtered search based on the suggested metadata filters from Stage 2. If the filtered results are insufficient (fewer than 3 chunks), it automatically falls back to an unfiltered search across the entire vector store. This ensures the system always returns enough context.
6. Noise filtering and deduplication (Stage 3):
We added post-retrieval cleaning that removes duplicate chunks and filters out weak or noisy chunks (e.g., page headers, chunks with too little text content). This was not present in the lab implementations.
7. LLM-based chunk scoring and reranking (Stage 4):
Lab 7's context compiler chain compressed the raw context in a single pass. Our context refinement stage scores each individual chunk on a 1–5 relevance scale using the LLM, reranks them by score, removes chunks below a quality threshold, and then compresses the top-ranked chunks into concise evidence blocks. This provides more fine-grained control over context quality.
8. Structured citations (Stage 5):
Lab 7 did not provide source citations. Our generation module includes structured citations referencing the top section, page numbers, and chunk ID for each piece of evidence used, improving answer transparency and verifiability.
9. Uncertainty handling (Stage 5):
When no relevant evidence is found or the highest-scoring chunk scores 2 or below, the system explicitly states that it cannot answer confidently rather than generating a potentially hallucinated response. This was not implemented in the lab.
10. Gradio interface (Stage 6):
Lab 7 used ipywidgets for the chat interface. We used Gradio's ChatInterface instead, which provides a cleaner conversation display with built-in message history visualization and works well in the Colab environment.

**Architecture Overview**

Stage 1: Ingestion & Indexing: This stage transforms the raw PDF document into a structured, searchable knowledge base.

*   PDF loading
*   section-aware parsing: The document is parsed based on its structural hierarchy (e.g., EXECUTIVE SUMMARY, SECTION I. THREATS), enabling structure-aware retrieval instead of flat page-based processing.
*   chunking within section: Each section is further divided into smaller chunks (e.g., 700 tokens with overlap), preserving semantic continuity while improving embedding quality.
*   metadata enrichment: Each section is further divided into smaller chunks (e.g., 700 tokens with overlap), preserving semantic continuity while improving embedding quality.
*   embedding
*   persist into Chroma

Stage 2: Query Understanding: This stage interprets the user query and transforms it into a structured query plan to improve retrieval precision by understanding the intent and context of the user query using an LLM.

*   classify query intent: The system categorizes the query into predefined types (e.g., threats, vulnerabilities, methodology), linking it to document structure.
*   optional query rewrite / expansion: The original query is rewritten into a more retrieval-friendly form and expanded with relevant terms to improve recall.
*   derive candidate metadata filters: Based on the intent, the system infers relevant metadata filters (e.g., top_section), narrowing the search space.

Stage 3: Retrieval: This stage retrieves candidate chunks from the vector store.
*   similarity search in Chroma: The system retrieves semantically similar chunks using vector similarity.
*   apply metadata filters when suitable: If strong structural signals exist (from Stage 2), metadata filters are applied to improve precision.
*   fetch larger candidate pool: A larger number of chunks (e.g., k=6) are retrieved to prioritize recall before refinement.

Stage 4: Context Refinement: If strong structural signals exist (from Stage 2), metadata filters are applied to improve precision.
*   compression / reranking: An LLM assigns relevance scores (1–5) to each chunk and reorders them.
*   remove redundant or weak chunks: An LLM assigns relevance scores (1–5) to each chunk and reorders them.
*   construct concise evidence context: Low-quality or duplicate chunks are removed.

Stage 5: Generation: Top-ranked chunks are compressed into shorter, high-signal evidence blocks.
*   grounded answer generation with gpt-4o-mini: The model generates answers strictly based on the provided evidence context.
*   cite chunk source / metadata: The system provides structured citations (e.g., section, page, chunk_id).
*   mention uncertainty when evidence insufficient: If evidence is weak or missing, the system explicitly states uncertainty instead of hallucinating.

Phase 6: Interface
*   A simple Gradio-based interface is implemented in Colab
*   Users input queries and receive answers with supporting evidence
*   The focus is on system transparency rather than UI complexity

**Stage 1: Ingestion & Indexing PDF loading bold**



In [ ]:
# reset database
if os.path.exists("/content/chroma_db_nmlra"):
    shutil.rmtree("/content/chroma_db_nmlra")

# section-aware parsing
PDF_PATH = "A2_42_source.pdf"
DOC_TITLE = "2024 National Money Laundering Risk Assessment"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

def clean_text(text: str) -> str:
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# chunking within section
top_section_pattern = re.compile(
    r'^(EXECUTIVE SUMMARY|INTRODUCTION|SECTION I\..+|SECTION II\..+|CONCLUSION|PARTICIPANTS|METHODOLOGY|TERMINOLOGY|LIST OF ACRONYMS)\s*$'
)

numbered_subtopic_pattern = re.compile(r'^\d+\.\s+.+$')

def is_top_section(line: str) -> bool:
    line = line.strip()
    return bool(top_section_pattern.match(line))

def is_numbered_subtopic(line: str) -> bool:
    line = line.strip()
    return bool(numbered_subtopic_pattern.match(line))

def is_topic_heading(line: str) -> bool:
    """
    识别像 Fraud / Drug Trafficking / Cybercrime 这种中层标题
    条件尽量保守，避免把普通句子误判成标题
    """
    line = line.strip()

    if not line:
        return False
    if is_top_section(line) or is_numbered_subtopic(line):
        return False
    if len(line) > 80:
        return False
    if line.endswith("."):
        return False
    if len(line.split()) > 8:
        return False

    # 首字母大写较多，适合标题
    words = line.split()
    capitalized_ratio = sum(1 for w in words if w[:1].isupper()) / max(len(words), 1)

    return capitalized_ratio >= 0.6

def parse_section_docs(pages: List[Document]) -> List[Document]:
    section_docs = []

    current_top_section = "FRONT_MATTER"
    current_topic = None
    current_subtopic = None
    current_text_lines = []
    current_start_page = 1

    def flush_buffer(end_page: int):
        nonlocal current_text_lines, current_start_page
        text = clean_text("\n".join(current_text_lines))
        if text:
            metadata = {
                "source": PDF_PATH,
                "doc_title": DOC_TITLE,
                "page_start": current_start_page,
                "page_end": end_page,
                "top_section": current_top_section,
                "topic": current_topic,
                "subtopic": current_subtopic,
            }
            section_docs.append(Document(page_content=text, metadata=metadata))
        current_text_lines = []

    for page in pages:
        page_num = page.metadata.get("page", 0) + 1
        text = clean_text(page.page_content)
        lines = [ln.strip() for ln in text.split("\n") if ln.strip()]

        for line in lines:
            # 顶层 section
            if is_top_section(line):
                flush_buffer(page_num)
                current_top_section = line
                current_topic = None
                current_subtopic = None
                current_start_page = page_num
                current_text_lines = [line]
                continue

            # 编号 subtopic，例如 1. Investment Fraud
            if is_numbered_subtopic(line):
                flush_buffer(page_num)
                current_subtopic = line
                current_start_page = page_num
                current_text_lines = [line]
                continue

            # topic，例如 Fraud / Drug Trafficking / Cybercrime
            if is_topic_heading(line):
                flush_buffer(page_num)
                current_topic = line
                current_subtopic = None
                current_start_page = page_num
                current_text_lines = [line]
                continue

            current_text_lines.append(line)

    flush_buffer(pages[-1].metadata.get("page", 0) + 1)
    return section_docs

section_docs = parse_section_docs(pages)
print(f"Created {len(section_docs)} section-level documents.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

# 看前几个 metadata 检查效果
for d in section_docs[:5]:
    print(d.metadata)
    print(d.page_content)
    print("-" * 80)

# metadata enrichment
def enrich_chunks(section_docs: List[Document]) -> List[Document]:
    final_chunks = []

    for sec_doc in section_docs:
        chunks = splitter.split_documents([sec_doc])

        for i, ch in enumerate(chunks):
            meta = dict(ch.metadata)
            section_name = (
                meta.get("subtopic")
                or meta.get("topic")
                or meta.get("top_section")
                or "unknown"
            )
            safe_section_name = re.sub(r'[^a-zA-Z0-9]+', '_', section_name)[:50]

            meta["chunk_id"] = f"{safe_section_name}_chunk_{i}"
            meta["doc_title"] = DOC_TITLE
            meta["content_type"] = "text"

            final_chunks.append(
                Document(
                    page_content=clean_text(ch.page_content),
                    metadata=meta
                )
            )

    return final_chunks

final_chunks = enrich_chunks(section_docs)

# embedding
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# persist into Chroma
vectorstore = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    collection_name="nmlra_2024_rag",
    persist_directory="/content/chroma_db_nmlra"
)

Created 353 section-level documents.
{'source': 'A2_42_source.pdf', 'doc_title': '2024 National Money Laundering Risk Assessment', 'page_start': 1, 'page_end': 1, 'top_section': 'FRONT_MATTER', 'topic': None, 'subtopic': None}
February 2024
--------------------------------------------------------------------------------
{'source': 'A2_42_source.pdf', 'doc_title': '2024 National Money Laundering Risk Assessment', 'page_start': 1, 'page_end': 1, 'top_section': 'FRONT_MATTER', 'topic': '2024 National Money', 'subtopic': None}
2024 National Money
--------------------------------------------------------------------------------
{'source': 'A2_42_source.pdf', 'doc_title': '2024 National Money Laundering Risk Assessment', 'page_start': 1, 'page_end': 1, 'top_section': 'FRONT_MATTER', 'topic': 'Laundering Risk', 'subtopic': None}
Laundering Risk
--------------------------------------------------------------------------------
{'source': 'A2_42_source.pdf', 'doc_title': '2024 National Money Laund

**Stage 2: Query Understanding**

In [ ]:
# Validation
VALID_TOP_SECTIONS = [
    "EXECUTIVE SUMMARY",
    "INTRODUCTION",
    "SECTION I. THREATS",
    "SECTION II. VULNERABILITIES AND RISKS",
    "CONCLUSION",
    "PARTICIPANTS",
    "METHODOLOGY",
    "TERMINOLOGY",
    "LIST OF ACRONYMS"
]

def validate_filters(filters: dict) -> dict:
    if "top_section" in filters:
        if filters["top_section"] not in VALID_TOP_SECTIONS:
            return {}
    return filters

# LLM-based intent classification
# LLM-based query rewrite / expansion
# LLM-based candidate metadata filter suggestion
query_understanding_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a query understanding module for a linear RAG system.

Your job is to analyze a user query about the PDF document:
"2024 National Money Laundering Risk Assessment"

The document has the following top-level sections:
- EXECUTIVE SUMMARY
- INTRODUCTION
- SECTION I. THREATS
- SECTION II. VULNERABILITIES AND RISKS
- CONCLUSION
- PARTICIPANTS
- METHODOLOGY
- TERMINOLOGY
- LIST OF ACRONYMS

You must do 4 things:
1. Classify the query intent into exactly one of these labels:
   - summary_overview
   - threats
   - vulnerabilities_risks
   - specific_topic_lookup
   - comparison_or_cross_section
   - methodology
   - terminology_acronyms
   - conclusion_or_implications

2. Rewrite the query into a retrieval-friendly query that better matches document language.

3. Suggest a few expanded retrieval terms.

4. Derive candidate metadata filters.
   Only use filters that are strongly justified.
   Prefer using "top_section".
   If no strong filter is appropriate, return an empty object {{}}.

Return ONLY valid JSON with this schema:
{{
  "intent": "...",
  "rewritten_query": "...",
  "expanded_terms": ["...", "..."],
  "candidate_filters": {{}},
  "reasoning": "..."
}}
Do not include markdown. Do not include any extra text.
"""
    ),
    (
        "human",
        "User query: {query}"
    )
])

query_understanding_chain = query_understanding_prompt | model | StrOutputParser()

def safe_json_loads(text: str):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        cleaned = text.strip()
        cleaned = cleaned.replace("```json", "").replace("```", "").strip()
        return json.loads(cleaned)

def understand_query_llm(query: str) -> dict:
    raw_output = query_understanding_chain.invoke({"query": query})

    parsed = safe_json_loads(raw_output)

    # 保底处理，避免字段缺失
    result = {
        "original_query": query,
        "intent": parsed.get("intent", "specific_topic_lookup"),
        "rewritten_query": parsed.get("rewritten_query", query),
        "expanded_terms": parsed.get("expanded_terms", []),
        "candidate_filters": validate_filters(parsed.get("candidate_filters", {})),
        "reasoning": parsed.get("reasoning", "")
    }
    return result

**Stage 3: Retrieval**

In [ ]:
def build_retrieval_query(plan: dict) -> str:
    rewritten = plan.get("rewritten_query", "").strip()
    expanded_terms = plan.get("expanded_terms", [])

    if expanded_terms:
        expansion_text = " | ".join(expanded_terms)
        return f"{rewritten}\nRelated terms: {expansion_text}"

    return rewritten


def deduplicate_docs(docs):
    seen = set()
    unique_docs = []

    for doc in docs:
        key = (
            doc.page_content.strip(),
            tuple(sorted(doc.metadata.items()))
        )
        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs

def is_weak_or_noisy_chunk(doc) -> bool:
    text = doc.page_content.strip()

    # 太短
    if len(text) < 80:
        return True

    # 明显页眉
    if text == "2024 ◆ National Money Laundering Risk Assessment":
        return True

    # 只有页眉 + 页码
    if "National Money Laundering Risk Assessment" in text and len(text.split()) < 10:
        return True

    # 几乎没有正文句子感
    alpha_chars = sum(c.isalpha() for c in text)
    if alpha_chars < 40:
        return True

    return False

def clean_retrieved_docs(docs):
    cleaned = []
    for doc in docs:
        if not is_weak_or_noisy_chunk(doc):
            cleaned.append(doc)
    return cleaned

def retrieve_with_optional_fallback(plan: dict, vectorstore, k: int = 8, min_results: int = 3):
    retrieval_query = build_retrieval_query(plan)
    filters = plan.get("candidate_filters", {})

    # First pass, similarity search in Chroma
    if filters:
        filtered_results = vectorstore.similarity_search(
            retrieval_query,
            k=k,
            filter=filters
        )
    else:
        filtered_results = vectorstore.similarity_search(
            retrieval_query,
            k=k
        )

    filtered_results = deduplicate_docs(filtered_results)
    filtered_results = clean_retrieved_docs(filtered_results)

    if len(filtered_results) >= min_results:
        return {
            "retrieval_mode": "filtered" if filters else "unfiltered",
            "retrieval_query": retrieval_query,
            "used_filters": filters,
            "results": filtered_results
        }

    # Fallback
    fallback_results = vectorstore.similarity_search(
        retrieval_query,
        k=k
    )
    fallback_results = deduplicate_docs(fallback_results)
    fallback_results = clean_retrieved_docs(fallback_results)

    return {
        "retrieval_mode": "fallback_unfiltered",
        "retrieval_query": retrieval_query,
        "used_filters": filters,
        "results": fallback_results
    }

**Stage 4: Context Refinement**

In [ ]:
rerank_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a context reranking and compression module for a linear RAG system.

Your task is to evaluate how useful a retrieved chunk is for answering a user query.

Score the chunk on a scale of 1 to 5:
- 5 = directly answers the query with highly relevant evidence
- 4 = strongly relevant and useful
- 3 = somewhat relevant but partial
- 2 = weakly relevant
- 1 = irrelevant or mostly noise

Return ONLY valid JSON in this format:
{{
  "score": 1,
  "reasoning": "...",
  "compressed_text": "..."
}}

Rules:
- compressed_text must keep only the most useful evidence from the chunk.
- compressed_text should be shorter than the original chunk.
- If the chunk is mostly irrelevant, compressed_text can be an empty string.
- Do not include markdown.
- Do not include extra text.
"""
    ),
    (
        "human",
        """
User query:
{query}

Query intent:
{intent}

Retrieved chunk metadata:
{metadata}

Retrieved chunk text:
{chunk_text}
"""
    )
])

rerank_chain = rerank_prompt | model | StrOutputParser()

# LLM-based reranker
def score_and_compress_chunk(query: str, query_plan: dict, doc):
    raw_output = rerank_chain.invoke({
        "query": query,
        "intent": query_plan.get("intent", ""),
        "metadata": json.dumps(doc.metadata, ensure_ascii=False),
        "chunk_text": doc.page_content
    })

    parsed = safe_json_loads(raw_output)

    score = parsed.get("score", 1)
    try:
        score = int(score)
    except:
        score = 1

    return {
        "doc": doc,
        "score": score,
        "reasoning": parsed.get("reasoning", ""),
        "compressed_text": parsed.get("compressed_text", "").strip()
    }

def rerank_and_compress_docs(query: str, query_plan: dict, retrieved_docs: list, top_n: int = 3):
    scored_docs = []

    for doc in retrieved_docs:
        try:
            scored = score_and_compress_chunk(query, query_plan, doc)
            scored_docs.append(scored)
        except Exception as e:
            # fallback: 如果某个 chunk 解析失败，保底保留原文但分数低一点
            scored_docs.append({
                "doc": doc,
                "score": 2,
                "reasoning": f"Fallback due to parsing error: {str(e)}",
                "compressed_text": doc.page_content[:500]
            })

    # 按分数从高到低排序
    scored_docs = sorted(scored_docs, key=lambda x: x["score"], reverse=True)

    # 只保留 compressed_text 非空的
    scored_docs = [x for x in scored_docs if x["compressed_text"]]

    # 取 top_n
    top_docs = scored_docs[:top_n]

    return top_docs

# context compression（top 3 chunks）
def build_concise_evidence_context(top_docs: list) -> str:
    context_blocks = []

    for i, item in enumerate(top_docs, 1):
        meta = item["doc"].metadata
        source_info = (
            f"[Evidence {i}] "
            f"top_section={meta.get('top_section', 'N/A')}, "
            f"page_start={meta.get('page_start', 'N/A')}, "
            f"chunk_id={meta.get('chunk_id', 'N/A')}"
        )

        block = f"{source_info}\n{item['compressed_text']}"
        context_blocks.append(block)

    return "\n\n".join(context_blocks)

def run_stage4_context_refinement(query: str, query_plan: dict, retrieval_output: dict, top_n: int = 3):
    retrieved_docs = retrieval_output["results"]

    top_docs = rerank_and_compress_docs(
        query=query,
        query_plan=query_plan,
        retrieved_docs=retrieved_docs,
        top_n=top_n
    )

    concise_context = build_concise_evidence_context(top_docs)

    return {
        "top_docs": top_docs,
        "concise_context": concise_context
    }

**Stage 5: Generation**

In [ ]:
# cite chunk source / metadata
def format_citations(top_docs: list) -> list:
    citations = []

    for item in top_docs:
        meta = item["doc"].metadata
        citation = {
            "top_section": meta.get("top_section", "N/A"),
            "page_start": meta.get("page_start", "N/A"),
            "page_end": meta.get("page_end", "N/A"),
            "chunk_id": meta.get("chunk_id", "N/A")
        }
        citations.append(citation)

    return citations

generation_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are the answer generation module in a linear RAG system.

Your job is to answer the user's question ONLY using the provided evidence context.

Rules:
1. Use only the evidence context.
2. Do not invent facts not supported by the evidence.
3. If the evidence is insufficient, say so clearly.
4. Write a clear, concise, well-structured answer.
5. When possible, mention the relevant section names naturally.
6. At the end, include a short "Evidence used" section that references the evidence labels.

Return only the answer text. Do not return JSON.
"""
    ),
    (
        "human",
        """
User question:
{query}

Query intent:
{intent}

Evidence context:
{context}
"""
    )
])

# grounded answer generation with gpt-4o-mini
generation_chain = generation_prompt | model | StrOutputParser()

def generate_grounded_answer(query: str, query_plan: dict, stage4_output: dict) -> dict:
    # cite chunk source / metadata
    concise_context = stage4_output["concise_context"]
    top_docs = stage4_output["top_docs"]

    # mention uncertainty when evidence insufficient
    if not top_docs:
        return {
            "answer": "I could not find enough relevant evidence in the document to answer this question confidently.",
            "citations": [],
            "grounded_context": concise_context,
            "insufficient_evidence": True
        }

    # mention uncertainty when evidence insufficient
    max_score = max(item["score"] for item in top_docs)

    if max_score <= 2:
        return {
            "answer": "The retrieved evidence is too weak to answer this question confidently based on the document.",
            "citations": format_citations(top_docs),
            "grounded_context": concise_context,
            "insufficient_evidence": True
        }

    # grounded answer generation with gpt-4o-mini
    answer = generation_chain.invoke({
        "query": query,
        "intent": query_plan.get("intent", ""),
        "context": concise_context
    })

    return {
        "answer": answer,
        "citations": format_citations(top_docs),
        "grounded_context": concise_context,
        "insufficient_evidence": False
    }

def print_generation_output(generation_output: dict):
    print("=== FINAL ANSWER ===")
    print(generation_output["answer"])

    print("\n=== CITATIONS ===")
    for i, c in enumerate(generation_output["citations"], 1):
        print(
            f"[Evidence {i}] "
            f"top_section={c['top_section']}, "
            f"page_start={c['page_start']}, "
            f"page_end={c['page_end']}, "
            f"chunk_id={c['chunk_id']}"
        )

**Integrate**

In [ ]:
def answer_question(query: str, vectorstore):
    # Stage 2
    query_plan = understand_query_llm(query)

    # Stage 3
    retrieval_output = retrieve_with_optional_fallback(
        plan=query_plan,
        vectorstore=vectorstore,
        k=6,
        min_results=3
    )

    # Stage 4
    stage4_output = run_stage4_context_refinement(
        query=query,
        query_plan=query_plan,
        retrieval_output=retrieval_output,
        top_n=3
    )

    # Stage 5
    generation_output = generate_grounded_answer(
        query=query,
        query_plan=query_plan,
        stage4_output=stage4_output
    )

    return {
        "query_plan": query_plan,
        "retrieval_output": retrieval_output,
        "stage4_output": stage4_output,
        "generation_output": generation_output
    }

In [ ]:
result = answer_question(
    "What are the main money laundering threats identified in the report?",
    vectorstore
)

print_generation_output(result["generation_output"])

=== FINAL ANSWER ===
The main money laundering threats identified in the report include:

1. **Predicate Crimes**: These are the underlying criminal activities that generate illicit proceeds, such as drug trafficking, fraud, human trafficking, and corruption.
2. **Professional Money Laundering (PML)**: This refers to organized efforts by individuals or groups to launder money on behalf of others.
3. **Transnational Criminal Organizations (TCOs)**: These groups engage in various criminal activities across borders, contributing significantly to money laundering.
4. **Vulnerabilities in Financial Institutions**: Some regulated financial institutions are identified as potential vulnerabilities for money laundering activities.
5. **Use of Cash**: Criminals and TCOs often prefer cash for laundering due to its anonymity and stability.

These threats collectively enable criminal activities and pose risks to U.S. national security.

Evidence used: [Evidence 1], [Evidence 2], [Evidence 3]

=== C

**UI**

In [ ]:
def format_result_for_display_verbose(result: dict) -> str:
    query_plan = result["query_plan"]
    retrieval_output = result["retrieval_output"]
    stage4_output = result["stage4_output"]
    generation_output = result["generation_output"]

    lines = []
    lines.append("Final Answer:")
    lines.append(generation_output["answer"])
    lines.append("")

    lines.append("Query Understanding:")
    lines.append(f"- Original query: {query_plan.get('original_query', '')}")
    lines.append(f"- Intent: {query_plan.get('intent', '')}")
    lines.append(f"- Rewritten query: {query_plan.get('rewritten_query', '')}")
    lines.append(f"- Expanded terms: {query_plan.get('expanded_terms', [])}")
    lines.append(f"- Candidate filters: {query_plan.get('candidate_filters', {})}")
    lines.append("")

    lines.append("Retrieval:")
    lines.append(f"- Retrieval mode: {retrieval_output.get('retrieval_mode', '')}")
    lines.append(f"- Retrieval query: {retrieval_output.get('retrieval_query', '')}")
    lines.append(f"- Used filters: {retrieval_output.get('used_filters', {})}")
    lines.append("")

    lines.append("Compressed Evidence Context:")
    lines.append(stage4_output.get("concise_context", ""))
    lines.append("")

    lines.append("Citations:")
    citations = generation_output.get("citations", [])
    if citations:
        for i, c in enumerate(citations, 1):
            lines.append(
                f"[Evidence {i}] "
                f"top_section={c.get('top_section', 'N/A')}, "
                f"page_start={c.get('page_start', 'N/A')}, "
                f"page_end={c.get('page_end', 'N/A')}, "
                f"chunk_id={c.get('chunk_id', 'N/A')}"
            )
    else:
        lines.append("No citations available.")

    return "\n".join(lines)

def chat_fn(user_query: str) -> str:
    if not user_query or not user_query.strip():
        return "Please enter a question."

    try:
        result = answer_question(user_query, vectorstore)
        return format_result_for_display_verbose(result)
    except Exception as e:
        return f"An error occurred while processing the query: {str(e)}"

In [ ]:
demo = gr.Interface(
    fn=chat_fn,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask a question about the PDF, e.g. What are the main money laundering threats identified in the report?",
        label="User Query"
    ),
    outputs=gr.Textbox(
        lines=20,
        label="RAG Assistant Response"
    ),
    title="RAG Assistant for the 2024 National Money Laundering Risk Assessment",
    description=(
        "This is a linear RAG system built with LangChain, Chroma, and gpt-4o-mini. "
        "It uses section-aware ingestion, LLM-based query understanding, metadata-filtered retrieval, "
        "context compression, and grounded answer generation."
    ),
    allow_flagging="never"
)

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c06ba6faf9ee0addbe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c06ba6faf9ee0addbe.gradio.live


In [ ]:
def chat_interface_fn(message, history):
    try:
        result = answer_question(message, vectorstore)
        return format_result_for_display_verbose(result)
    except Exception as e:
        return f"An error occurred while processing the query: {str(e)}"

demo = gr.ChatInterface(
    fn=chat_interface_fn,
    title="Linear RAG Assistant",
    description="Ask questions about the 2024 National Money Laundering Risk Assessment."
)

demo.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2c0b60bc08980bb348.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
